In [5]:
%pip install yagmail

Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install langchain_google_genai langgraph langchain

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from dotenv import load_dotenv
import os
import yagmail
load_dotenv()
groq_api_key=os.getenv("GROQ_API_KEY")
serp_api_key=os.getenv("SERP_API_KEY")
app_password=os.getenv("APP_PASSWORD")
google_generative_ai_api=os.getenv("Google_Generative_AI_API")
email=os.getenv("EMAIL")
yag = yagmail.SMTP(email,password=app_password)
#print(f"email {email} and password {app_password} loaded successfully")
# yag.send(
#     to="email@gmail.com",
#     subject="Test Email",
#     contents="Hello, this is sent using Yagmail!"
# )

# print("Email sent!")


Email sent!


In [14]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

llm=ChatGoogleGenerativeAI(
    api_key=google_generative_ai_api,
    temperature=0.5,
    model="gemini-flash-lite-latest",
    max_tokens=None,
    max_retries=2,
    timeout=None
)

In [ ]:
from langchain_core.tools import tool
from serpapi import GoogleSearch
from langgraph.checkpoint.memory import InMemorySaver
# @tool decorator and a proper triple-quoted docstring
@tool
def send_email_tool(recipient: str, subject: str, contents: str) -> str:
    """Sends an email to the recipient with the specified subject and contents."""
    yag.send(
        to=recipient,
        subject=subject,
        contents=contents
    )
    return "Email sent!"
#lets add Serp Search in it 
def serpapi_search(query: str):             #Query is expected to be a str thats type hints useful for verification in return types.
    """Searches for a query using the SerpAPI on Google."""
    params = {
        "q": query,
        "hl": "en",
        "gl": "us",
        "api_key": serp_api_key           #pydantic verification due to Type Hints 
    }
    search = GoogleSearch(params)
    results = search.get_dict()
    
    # Extract top results (titles + links)
    if "organic_results" in results:
        return [
            {"title": r["title"], "link": r["link"], "snippet": r.get("snippet", "")}
            for r in results["organic_results"][:5]       #:5 gets only 0:5 means 5 items from results dictionary 
        ]
    return {"error": "No results found"}

memory = InMemorySaver()
# Create your agent
graph = create_agent(
    name="Email Writing Agent",
    model=llm,
    tools=[send_email_tool, serpapi_search],
    system_prompt="You are an email assistant. Use send_email_tool to send emails (requires valid recipient, subject, contents). Use serpapi_search to research facts or context for emails if needed. Be concise and professional in your email drafts. I'm Muneeb",
    checkpointer=memory
)


In [54]:
from langchain_core.messages import HumanMessage
contents= input("Please tell me who do you wanna send email to and what do you want to say in the email?")
inputs ={ "messages":
        [HumanMessage(content=contents)] 
        }

config= {"configurable": {"thread_id": "email_agent_session_123"}}
for chunk in graph.stream(inputs,config=config, stream_mode="updates"):  # type: ignore 
    print(chunk)
    

{'model': {'messages': [AIMessage(content=[], additional_kwargs={'function_call': {'name': 'serpapi_search', 'arguments': '{"query": "best fuel efficient used sedans crossovers cars pakistan price"}'}, '__gemini_function_call_thought_signatures__': {'5sCky7bk': 'EjQKMgERTTIPdDzhjF/HCRZx/nJbEZZAVKO8tEWqOcTNeveC/yh/KSOdTJ9Wr3F9I83Npzvr'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, name='Email Writing Agent', id='lc_run--019fbcea-c12e-7543-a390-016ba7a61f06-0', tool_calls=[{'name': 'serpapi_search', 'args': {'query': 'best fuel efficient used sedans crossovers cars pakistan price'}, 'id': '5sCky7bk', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 4645, 'output_tokens': 27, 'total_tokens': 4672, 'input_token_details': {'cache_read': 0}})]}}
{'tools': {'messages': [ToolMessage(content='[{"title": "What is a good crossover car to buy in Pakistan within ...", "link"